# Phase 2 — Patent Similarity Prototype (Evaluation)

Validates the Phase 2 retrieval design on a small hand-written toy corpus
before the real Phase 1 G06T dataset is available.

Pipeline code lives in `src/embedding_pipeline.py`. Toy data lives in
`data/toy_patents.py`. This notebook only defines the locked evaluation
queries and runs/reports the evaluation.

Two-stage design: Stage 1 abstract-embedding search (Gemini
`gemini-embedding-001` + FAISS `IndexFlatIP`) → top candidates → Stage 2
claims-embedding rerank. `final_score = 0.3 * abstract_score + 0.7 * claim_score`.

In [1]:
import sys

sys.path.append("..")

from data.toy_patents import TOY_PATENTS

from src.phase2_embedding_retrieval.embedding_pipeline import (
    get_client,
    build_indexes,
    evaluate_query,
    print_ranking,
)

client = get_client()

index_data = build_indexes(client, TOY_PATENTS)

print("Abstract embeddings:", index_data["abstract_embeddings"].shape)

print(
    "Vectors in FAISS index:",
    index_data["abstract_index"].ntotal,
)

print(
    "Toy patents:",
    len(TOY_PATENTS),
)

print(
    "Missing claims:",
    sum(patent["claims"] is None for patent in TOY_PATENTS),
)


Abstract embeddings: (8, 3072)
Vectors in FAISS index: 8
Toy patents: 8
Missing claims: 3


## Locked evaluation queries

Predictions were fixed before running the evaluation.

In [2]:
eval_queries = [
    {"id": "Q1", "query": "A deep learning system receives digital images and uses multiple convolutional layers to classify each image into a predefined category.", "expected": "P001"},
    {"id": "Q2", "query": "A neural network analyzes MRI and CT medical images to identify abnormal regions and detect possible tumors.", "expected": "P002"},
    {"id": "Q3", "query": "A camera-based deep learning system detects pedestrians, vehicles, and other road objects for an autonomous vehicle.", "expected": "P003"},
    {"id": "Q4", "query": "A transformer-based natural language processing system reads a document and automatically generates a concise summary containing its most important information.", "expected": "P004"},
    {"id": "Q5", "query": "A security system captures a person's face, extracts facial features, compares them with stored identities, and grants access to an authorized person.", "expected": "P005"},
    {"id": "Q6", "query": "A deep learning vision system analyzes camera images of vehicles and other objects, identifies the detected entities, and uses the predicted identity and location of each object to support real-time vehicle operation.", "expected": "P003"},
    {"id": "Q7", "query": "A system processes audio recordings to identify spoken sounds and classify different acoustic signals.", "expected": "None"},
    {"id": "Q8", "query": "A method receives an image, processes it through multiple convolutional layers, and generates a classification label for the image.", "expected": "P001"}
]

## Run Stage 1 + Stage 2 evaluation

Q6 (the deliberate ambiguity stress test) is printed first.

In [3]:
patents = index_data["patents"]
evaluation_results = []

for item in eval_queries:
    result = evaluate_query(
        client,
        index_data,
        item["query"],
    )

    print("\n" + "=" * 90)
    print(f"{item['id']}")
    print(f"Query: {item['query']}")
    print(f"Expected: {item['expected']}")

    print("\n--- STAGE 1: ABSTRACT-ONLY ---")
    print_ranking(
        patents,
        result["stage1_indices"],
        result["abstract_scores"],
    )

    print("\n--- STAGE 2: CLAIMS RERANKED (30/70) ---")
    print_ranking(
        patents,
        result["stage2_indices"],
        result["final_scores"],
    )

    evaluation_results.append({
        "query_id": item["id"],
        "expected": item["expected"],
        "stage1_ranking": [
            patents[i]["id"]
            for i in result["stage1_indices"]
        ],
        "stage2_ranking": [
            patents[i]["id"]
            for i in result["stage2_indices"]
        ],
    })


Q1
Query: A deep learning system receives digital images and uses multiple convolutional layers to classify each image into a predefined category.
Expected: P001

--- STAGE 1: ABSTRACT-ONLY ---
Rank 1
Patent ID: P001
Title: Image Classification Using Convolutional Neural Networks
Score: 0.8275
--------------------------------------------------
Rank 2
Patent ID: TOY-MISSING-CLAIMS-02
Title: Neural Network System for Detecting Tumors
Score: 0.7058
--------------------------------------------------
Rank 3
Patent ID: P005
Title: Facial Recognition Based Access Control
Score: 0.6989
--------------------------------------------------
Rank 4
Patent ID: TOY-MISSING-CLAIMS-03
Title: Image Processing System for Medical Diagnosis
Score: 0.6830
--------------------------------------------------
Rank 5
Patent ID: P002
Title: Medical Image Tumor Detection System
Score: 0.6762
--------------------------------------------------
Rank 6
Patent ID: TOY-MISSING-CLAIMS-01
Title: Medical Image Segmentation

## Prototype findings

See `docs/findings/phase2_findings.md` for the full write-up. Summary:

- Stage 1 retrieved the expected patent at rank 1 for all seven in-domain sanity-check queries.
- Stage 2 did not flip a correct rank-1 result into an incorrect one in this test set.
- The Q6 ambiguity test showed claims reranking increased the P003–P005 score separation.
- Q7 exposed a missing no-match / low-confidence mechanism: nearest-neighbor search always returns a closest result even when no patent is a strong match.